In [1]:
# V2: Multi-pooling, Multi-sample dropout, Weighted layer pooling, DeBERTa-v3-base
from transformers import AutoTokenizer, AutoModel, AutoConfig, PreTrainedModel, Trainer, TrainingArguments
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from sklearn.model_selection import train_test_split
from src.preprocess import QuestPreprocessor
from src.postprocess import QuestPostprocessor
from src.loss import SpearmanLoss
import os

# Switch to DeBERTa-v3-base for better ranking performance
MODEL_NAME = "answerdotai/ModernBERT-base"

TARGET_COLS = ['question_asker_intent_understanding',
 'question_body_critical',
 'question_conversational',
 'question_expect_short_answer',
 'question_fact_seeking',
 'question_has_commonly_accepted_answer',
 'question_interestingness_others',
 'question_interestingness_self',
 'question_multi_intent',
 'question_not_really_a_question',
 'question_opinion_seeking',
 'question_type_choice',
 'question_type_compare',
 'question_type_consequence',
 'question_type_definition',
 'question_type_entity',
 'question_type_instructions',
 'question_type_procedure',
 'question_type_reason_explanation',
 'question_type_spelling',
 'question_well_written',
 'answer_helpful',
 'answer_level_of_information',
 'answer_plausible',
 'answer_relevance',
 'answer_satisfaction',
 'answer_type_instructions',
 'answer_type_procedure',
 'answer_type_reason_explanation',
 'answer_well_written']

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class QuestDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['text_input'])

        # Tokenize
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze() for k, v in enc.items()}

        if TARGET_COLS[0] in row:
            labels = torch.tensor(row[TARGET_COLS].astype(float).values, dtype=torch.float32)
            item["labels"] = labels

        return item


class MultiPooling(nn.Module):
    """Combine [CLS], mean, and max pooling for richer representations."""
    def forward(self, hidden_states, attention_mask):
        # [CLS] token
        cls_output = hidden_states[:, 0, :]
        
        # Mean pooling (mask-aware)
        mask = attention_mask.unsqueeze(-1).float()
        mean_output = (hidden_states * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        
        # Max pooling (mask out padding with large negative)
        hidden_masked = hidden_states.clone()
        hidden_masked[attention_mask == 0] = -1e9
        max_output = hidden_masked.max(dim=1)[0]
        
        return torch.cat([cls_output, mean_output, max_output], dim=-1)


class WeightedLayerPooling(nn.Module):
    """Learn to weight and combine multiple transformer layers."""
    def __init__(self, n_layers, layer_start=4):
        super().__init__()
        self.layer_start = layer_start
        n_used_layers = n_layers - layer_start
        self.layer_weights = nn.Parameter(torch.ones(n_used_layers) / n_used_layers)
    
    def forward(self, all_hidden_states):
        # all_hidden_states: tuple of (n_layers+1, batch, seq, hidden)
        layers = torch.stack(all_hidden_states[self.layer_start:], dim=0)
        weights = F.softmax(self.layer_weights, dim=0)
        weighted = (layers * weights.view(-1, 1, 1, 1)).sum(dim=0)
        return weighted


class CustomModelV2(PreTrainedModel):
    """
    Improved model with:
    - Weighted layer pooling (learns which layers matter)
    - Multi-pooling (CLS + mean + max)
    - Multi-sample dropout (5 dropout masks averaged during training)
    """
    def __init__(self, config, n_targets=30, n_dropouts=5, dropout_prob=0.1, 
                 layer_start=4, freeze_encoder=False):
        super().__init__(config)
        
        # Encoder
        if config._name_or_path:
            self.encoder = AutoModel.from_pretrained(config._name_or_path)
        else:
            self.encoder = AutoModel.from_config(config)
        
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

        # Weighted layer pooling
        n_layers = config.num_hidden_layers + 1  # +1 for embedding layer
        self.weighted_layer_pool = WeightedLayerPooling(n_layers, layer_start)
        
        # Multi-pooling
        self.multi_pool = MultiPooling()
        
        # Multi-sample dropout
        self.n_dropouts = n_dropouts
        self.dropouts = nn.ModuleList([nn.Dropout(dropout_prob) for _ in range(n_dropouts)])
        
        # Regressor: input is 3x hidden_size due to multi-pooling
        self.regressor = nn.Linear(config.hidden_size * 3, n_targets)
        
        self.spearman_loss = SpearmanLoss()
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None, token_type_ids=None):
        # Get all hidden states
        outputs = self.encoder(
            input_ids=input_ids, 
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        
        # Weighted layer pooling
        weighted_hidden = self.weighted_layer_pool(outputs.hidden_states)
        
        # Multi-pooling (CLS + mean + max)
        pooled = self.multi_pool(weighted_hidden, attention_mask)
        
        # Multi-sample dropout during training
        if self.training:
            raw_logits = torch.mean(torch.stack([
                self.regressor(dp(pooled)) for dp in self.dropouts
            ]), dim=0)
        else:
            raw_logits = self.regressor(pooled)
        
        preds = torch.sigmoid(raw_logits)
        
        loss = None
        if labels is not None:
            loss_bce = nn.BCEWithLogitsLoss()(raw_logits, labels)
            loss_spearman = self.spearman_loss(preds, labels)
            loss = loss_bce * 0.9 + loss_spearman * 0.1

        return {"loss": loss, "logits": preds}


config = AutoConfig.from_pretrained(MODEL_NAME)
model = CustomModelV2(config, n_targets=len(TARGET_COLS))

In [2]:
# --- TRAINING BLOCK ---
import numpy as np
from scipy.stats import spearmanr

# 1. Load Train Data
print('Loading train.csv...')
raw_train_df = pd.read_csv('train.csv')

# --- NEW PREPROCESSING STEP ---
preprocessor = QuestPreprocessor(use_category=True, use_host=True)
train_df_processed = preprocessor.preprocess(raw_train_df)
# ------------------------------

postprocessor = QuestPostprocessor(train_labels=raw_train_df[TARGET_COLS].values)


train_split, val_split = train_test_split(train_df_processed, test_size=0.1, random_state=42)

train_ds = QuestDataset(train_split, tokenizer)
val_ds = QuestDataset(val_split, tokenizer)

# 2. Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    logits = postprocessor.optimize_ranks(logits)
    
    rhos = []
    for i in range(labels.shape[1]):
        if np.std(labels[:, i]) == 0 or np.std(logits[:, i]) == 0:
            rhos.append(0.0)
        else:
            rho, _ = spearmanr(labels[:, i], logits[:, i])
            rhos.append(rho)
    return {'spearman': np.nanmean(rhos)}

# 3. Train (adjusted for DeBERTa-v3 + larger model)
training_args = TrainingArguments(
    output_dir='./results_v2',
    num_train_epochs=15,
    per_device_train_batch_size=16,  # Reduce if OOM
    per_device_eval_batch_size=16,
    learning_rate=2e-5,  # Slightly lower LR for DeBERTa
    warmup_ratio=0.1,    # Use ratio instead of fixed steps
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='spearman',
    greater_is_better=True,
    report_to='none',
    fp16=True,  # Set True if using CUDA GPU for faster training
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print('Starting Training...')
trainer.train()
print('Training Complete. Evaluating...')
eval_res = trainer.evaluate()
print(f'Validation Spearman Rho: {eval_res["eval_spearman"]:.4f}')

# 4. SAVE MODEL FOR INFERENCE
output_model_dir = './saved_model_v2'
if not os.path.exists(output_model_dir):
    os.makedirs(output_model_dir)

# Save the model weights
trainer.save_model(output_model_dir)
# Save the tokenizer
tokenizer.save_pretrained(output_model_dir)
# Save the config
config.save_pretrained(output_model_dir)
print(f"Model, tokenizer, and config saved to {output_model_dir}")

Loading train.csv...
Starting Training...


Epoch,Training Loss,Validation Loss,Spearman
1,0.512900,0.458356,0.216071
2,0.446700,0.426957,0.322043
3,0.414900,0.406846,0.360821
4,0.397300,0.400563,0.385223
5,0.375500,0.399694,0.392739
6,0.367400,0.398206,0.394860
7,0.346700,0.398389,0.389685
8,0.334800,0.401242,0.385500


KeyboardInterrupt: 